# Stitching consistency at 3x3 FOV neighbourhoods

Revisits the camera-rotation/BigStitcher-style correction work
(`acquisition/camera_rotation.py`, `misc/correct_camera_rotation.ipynb`, see
`prompt_history/2026_07_3*_camera_rotation_*.md` and
`2026_08_01_0110_implement_per_fov_global_position_optimization.md`). That
work concluded a single global affine transform cannot correct real per-FOV
positioning jitter -- fixing it needs a genuine BigStitcher-style joint
position solve (`fit_global_positions`) -- but that solve's own sparse anchor
sampling mostly produces small disconnected "star" correspondence-graph
components: one anchor + up to 4 leaves, each leaf constrained by exactly ONE
measurement, so a solved leaf position is numerically identical to that one
correspondence's own measurement. There is no redundant, self-checking
measurement per FOV yet.

This notebook looks at that redundancy question directly, at a small,
inspectable scale, instead of across a whole experiment:

1. **4-connected alignment** -- register a centre FOV's up/down/left/right
   neighbours to it via overlap-border phase correlation
   (`camera_rotation.register_neighbor_pair`, reusing the same primitive the
   correction pipeline itself uses).
2. **Corner concordance** -- at each of the centre FOV's 4 corners, the
   corner region is implied by TWO different registrations (its "up"
   neighbour's own alignment, and its "left" neighbour's own alignment, for
   the up-left corner). If the grid were perfectly rigid these would agree
   exactly; comparing them directly measures how much they don't.
3. **Diagonal loop closure** -- for each corner's diagonal FOV, register it
   via BOTH of the corner's already-aligned 4-connected neighbours
   independently (e.g. the up-left diagonal FOV is registered once through
   "up", once through "left") and compare the two resulting positions -- the
   classic BigStitcher loop-closure check: a consistent grid should give the
   same answer both ways.
4. Do 1-3 on **both the bead channel** (frame 0 -- HAL's own fiducial/
   focus-lock reference frame) **and the DAPI channel** (a configurable
   z-index), and compare the two channels' own registrations against each
   other -- do two independent signal sources agree on the same real
   misalignment?

Repeated for **3 independent (non-overlapping) 3x3 neighbourhoods**, each
centred on a randomly-chosen FOV near the tissue centroid, so no single
neighbourhood's own idiosyncrasies (weak signal, a real local defect) drives
the whole conclusion.

**Why no SLURM submission machinery here** (unlike
`07_cluster_submit_analysis.ipynb`/`cli_analyze_fov.py`): this notebook reads
at most 3 neighbourhoods x 9 FOVs x 2 frames (bead + one DAPI z-plane) = 54
small, targeted reads -- a tiny fraction of that pipeline's own per-FOV cost
(a full multi-frame z-stack, budgeted at 2h/SLURM-task). The intent is that
this notebook's own Jupyter kernel already runs ON the cluster (e.g. a login
or interactive compute-node session) where the real data lives, reading
directly -- no separate job submission adds anything here.

**Verification note**: this notebook was authored and verified end-to-end
against a small SYNTHETIC fixture (fake multi-FOV `.zarr` stacks with known,
injected shifts -- see `cache/scripts/build_test_stitching_fixture.py`,
gitignored), since no real cluster dataset was reachable from the authoring
environment. The synthetic run confirms the registration/corner/diagonal
logic executes correctly end-to-end and recovers injected shifts (see the
notebook's own printed sanity checks below) -- it does NOT confirm anything
about a real dataset's real camera-vs-stage alignment. Review the results
critically on real data before drawing conclusions from them.

**Outputs**: `analysis/cache/test_stitching/*.csv` (cardinal/corner/diagonal
result tables) and `analysis/figures/test_stitching.*.png` (overview +
per-neighbourhood corner/diagonal overlay figures).

## 1 — Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial import KDTree
from scipy.ndimage import shift as ndi_shift

# notebooks/tests/ is two levels under the repo root (MERci/), same
# convention as notebooks/misc/ and notebooks/during_imaging/.
MERCI_DIR = Path(os.getcwd()).parent.parent   # MERci/

sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.io              import load_positions, read_image_frames
from MERci.common.experiment_info import load_experiment_info, resolve_sample_identity
from MERci.acquisition.configs    import (
    find_frame_table_for_hal_config, get_camera_pixel_size_um, get_camera_frame_size,
    get_all_color_frame_indices,
)
from MERci.acquisition.positions       import find_grid_neighbor
from MERci.acquisition.camera_rotation import (
    apply_microscope_orientation, crop_overlap, register_neighbor_pair,
)
from MERci.acquisition.alignment  import remove_hot_pixels, phase_drift
from MERci.acquisition.merlin_config import load_microscope_orientation
from MERci.progress_display       import ProgressReporter

NOTEBOOK_NAME = "test_stitching"
print(f"MERCI_DIR : {MERCI_DIR}")

## 2 — Parameters

In [ ]:
# Set this to analyze a DIFFERENT experiment than the one this MERci clone
# physically lives in (e.g. this clone is a shared/dev copy used to point at
# various real datasets on the cluster). None (default) auto-detects the
# normal way: SAMPLE_DIR = MERCI_DIR.parent, i.e. this clone lives inside the
# experiment folder it analyzes, same as every other notebook in this repo.
DATASET_DIR_OVERRIDE = None

SAMPLE_DIR = Path(DATASET_DIR_OVERRIDE) if DATASET_DIR_OVERRIDE is not None else MERCI_DIR.parent

# resolve_sample_identity only looks at path component NAMES (no existence
# check needed) -- appending a synthetic "MERci" gives the same split/flat
# resolution logic it uses for the normal (auto-detected) case, even when
# SAMPLE_DIR was overridden to a folder that doesn't actually contain a
# MERci clone at all.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(SAMPLE_DIR / "MERci")
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

info       = load_experiment_info(SAMPLE_DIR / "metadata" / "experiment_info.yaml")
MICROSCOPE = info.microscope

# The bead (fiducial/focus-lock) frame is always the raw camera frame at
# index 0 of the cells-round stack -- HAL's own convention (see CLAUDE.md's
# "beads (frame 0)" / EXCLUDED_COLORS discussion in round_mosaics.ipynb).
BEAD_FRAME_INDEX = 0

# DAPI channel + z-INDEX (not a z position in um) -- the 4th z-step (0-indexed)
# of DAPI_COLOR_NM's own z-sweep in the cells-round frame table, mirroring
# "beads (frame 0)"'s own index convention. Adjust DAPI_COLOR_NM if this
# experiment's cells round doesn't use 405nm for DAPI.
DAPI_COLOR_NM = 405.0
DAPI_Z_INDEX  = 3

# How many independent (non-overlapping) 3x3 neighbourhoods to analyze.
N_NEIGHBORHOODS = 3

# Candidate centre FOVs are drawn from this fraction of all cells-round FOVs
# closest to the tissue centroid (i.e. genuinely "near the centre of the
# tissue", not just anywhere with a complete 3x3 neighbourhood), then
# shuffled -- deterministic by default (SEED fixed) for reproducible re-runs;
# set SEED=None for a fresh random pick each run.
CENTRAL_CANDIDATE_FRACTION = 0.3
SEED = 0

TOLERANCE_FRACTION = 0.25   # same default as find_grid_neighbor/find_exterior_fovs
UPSAMPLE_FACTOR     = 10    # sub-pixel registration precision (1/UPSAMPLE_FACTOR px)

FORCE_RECOMPUTE = False   # set True to re-register even if a cache already exists

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- same values used
# throughout this repo's other notebooks.
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"SAMPLE_DIR  : {SAMPLE_DIR}")
print(f"SAMPLE_NAME : {SAMPLE_NAME}  (IMAGING_DIR={IMAGING_DIR!r})")
print(f"MICROSCOPE  : {MICROSCOPE}")

## 3 — Resolve the cells round + FOV geometry

Same pattern as `misc/correct_camera_rotation.ipynb` section 3: build
`ExperimentMetadata` for the cells round, then measure `STEP_SIZE_UM`/
`OVERLAP_FRACTION` from the REAL positions file (median nearest-neighbour
distance) rather than trusting `ExperimentConfig`'s own
`pixel_size_um`/`image_size_px`/`non_overlap_fraction` formula, which that
notebook confirmed can silently disagree with the real camera/grid and break
every registration.

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)


def resolve_round_id(meta, imaging_type):
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


CELLS_ROUND_ID = resolve_round_id(meta, "cells")
if not meta.round_fully_written(CELLS_ROUND_ID):
    print(f"WARNING: cells round {CELLS_ROUND_ID} is not yet fully written on disk -- "
          f"some FOVs sampled below may be missing.")

cells_series = next(s for s in meta.series_for_round(CELLS_ROUND_ID) if s.hal_config)
hal_path     = Path(config.settings_dir) / cells_series.hal_config
ft_path      = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
if ft_path is None or not ft_path.exists():
    raise FileNotFoundError(f"No frame table found for the cells round (hal_config={hal_path}).")
frame_table = pd.read_csv(ft_path, index_col=0)

dapi_frame_indices = get_all_color_frame_indices(frame_table, DAPI_COLOR_NM)
if DAPI_Z_INDEX >= len(dapi_frame_indices):
    raise ValueError(
        f"DAPI_Z_INDEX={DAPI_Z_INDEX} out of range -- {DAPI_COLOR_NM:.0f}nm only has "
        f"{len(dapi_frame_indices)} z-plane(s) in this frame table.")
DAPI_FRAME_INDEX = dapi_frame_indices[DAPI_Z_INDEX]

print(f"Cells round        : {CELLS_ROUND_ID}  (series pattern: {cells_series.name!r})")
print(f"Frame table         : {ft_path}")
print(f"Bead frame          : index {BEAD_FRAME_INDEX}  "
      f"(color={frame_table.loc[BEAD_FRAME_INDEX, 'color']}, z={frame_table.loc[BEAD_FRAME_INDEX, 'z']}) "
      f"-- confirm this is really the bead/fiducial frame on real data.")
print(f"DAPI frame          : index {DAPI_FRAME_INDEX}  (z-index {DAPI_Z_INDEX} of "
      f"{len(dapi_frame_indices)} {DAPI_COLOR_NM:.0f}nm z-plane(s), z={frame_table.loc[DAPI_FRAME_INDEX, 'z']})")

PIXEL_SIZE_UM     = get_camera_pixel_size_um(MICROSCOPE)
FRAME_WIDTH_PX, _ = get_camera_frame_size(MICROSCOPE)
FRAME_WIDTH_UM    = FRAME_WIDTH_PX * PIXEL_SIZE_UM

full_positions = load_positions(config.positions_txt)
cells_fov_ids  = sorted(f for f in full_positions if f in meta.fovs)

coords_arr   = np.array([full_positions[f] for f in cells_fov_ids], dtype=float)
nn_dist, _   = KDTree(coords_arr).query(coords_arr, k=2)
STEP_SIZE_UM = float(np.median(nn_dist[:, 1]))
OVERLAP_FRACTION = max(0.0, 1.0 - STEP_SIZE_UM / FRAME_WIDTH_UM)

print(f"\nPixel size (um)    : {PIXEL_SIZE_UM}")
print(f"Frame width (um)   : {FRAME_WIDTH_UM:.3f}  ({FRAME_WIDTH_PX} px)")
print(f"Step size (um)     : {STEP_SIZE_UM:.3f}  (measured, median nearest-neighbour distance)")
print(f"Overlap fraction   : {OVERLAP_FRACTION:.3f}")
print(f"FOVs in cells round: {len(cells_fov_ids)}")

## 4 — Microscope orientation

Reads this microscope's verified `transpose`/`flip_horizontal`/
`flip_vertical` convention straight from its MERlin microscope-parameters
JSON (`data/configs/merlin/microscope/*.json`), same as
`misc/correct_camera_rotation.ipynb` section 4 -- every raw frame is
reoriented via `apply_microscope_orientation` immediately after reading,
before any registration.

In [ ]:
orientation = load_microscope_orientation(MICROSCOPE, MERCI_DIR / "data" / "configs" / "merlin" / "microscope")
ORIENT_TRANSPOSE       = bool(orientation.get("transpose", False))
ORIENT_FLIP_HORIZONTAL = bool(orientation.get("flip_horizontal", False))
ORIENT_FLIP_VERTICAL   = bool(orientation.get("flip_vertical", False))

# NOTE: orientation is applied exactly ONCE, inside load_channel_frames below --
# every frame this notebook ever registers/crops is already oriented by the
# time it's used, so register_neighbor_pair/register_chained below are called
# with NO orient_* kwargs (they default to False/no-op). Passing the same
# orient_transpose/flip_* flags there too would re-apply apply_microscope_
# orientation a SECOND time on an already-oriented image -- not a no-op
# (transpose+flip is not idempotent) -- which silently corrupts every
# registration into measuring garbage.

print(f"transpose={ORIENT_TRANSPOSE}  flip_horizontal={ORIENT_FLIP_HORIZONTAL}  "
      f"flip_vertical={ORIENT_FLIP_VERTICAL}")


def load_channel_frames(fov_id):
    '''{'beads': oriented bead frame, 'dapi': oriented DAPI frame} for one FOV --
    one file open, both frames read together (read_image_frames is selective).'''
    path = cells_series.resolve_path(fov_id, config.image_suffix)
    bead_raw, dapi_raw = read_image_frames(
        path, [BEAD_FRAME_INDEX, DAPI_FRAME_INDEX],
        frame_width=config.frame_width, frame_height=config.frame_height,
    )
    return {
        "beads": apply_microscope_orientation(bead_raw, ORIENT_TRANSPOSE, ORIENT_FLIP_HORIZONTAL, ORIENT_FLIP_VERTICAL),
        "dapi":  apply_microscope_orientation(dapi_raw, ORIENT_TRANSPOSE, ORIENT_FLIP_HORIZONTAL, ORIENT_FLIP_VERTICAL),
    }


CHANNELS = ("beads", "dapi")

## 5 — Find 3 independent 3x3 (8-connected) neighbourhoods

`find_3x3_block` (ported from `misc/correct_camera_rotation.ipynb` section
10) finds a centre FOV's full 3x3, 8-connected neighbourhood -- 4 cardinal
neighbours plus their shared diagonal FOVs (each diagonal resolved via
EITHER of its two cardinal neighbours, cross-checked against each other).
Candidate centres are the FOVs closest to the tissue centroid (a genuine
"near the centre of the tissue" pool), shuffled by `SEED` for a random pick;
each accepted neighbourhood's 9 FOVs are removed from the candidate pool
before picking the next one, so the 3 neighbourhoods never share a FOV.

In [ ]:
def find_3x3_block(fov_ids, positions, step_size_um, tolerance_fraction):
    '''First complete 3x3 (8-connected) FOV neighbourhood found among fov_ids --
    a centre FOV plus all 8 of its neighbours (4-connected + diagonal). Returns
    a dict keyed by compass direction (plus "center"), or None.'''
    def _diag(fov_a, dir_from_a, fov_b, dir_from_b):
        d = find_grid_neighbor(fov_a, positions, dir_from_a, step_size_um, tolerance_fraction)
        if d is None:
            d = find_grid_neighbor(fov_b, positions, dir_from_b, step_size_um, tolerance_fraction)
        return d

    for center in fov_ids:
        up    = find_grid_neighbor(center, positions, "up",    step_size_um, tolerance_fraction)
        down  = find_grid_neighbor(center, positions, "down",  step_size_um, tolerance_fraction)
        left  = find_grid_neighbor(center, positions, "left",  step_size_um, tolerance_fraction)
        right = find_grid_neighbor(center, positions, "right", step_size_um, tolerance_fraction)
        if None in (up, down, left, right):
            continue
        up_left    = _diag(up, "left", left, "up")
        up_right   = _diag(up, "right", right, "up")
        down_left  = _diag(down, "left", left, "down")
        down_right = _diag(down, "right", right, "down")
        if None in (up_left, up_right, down_left, down_right):
            continue
        return {
            "center": center, "up": up, "down": down, "left": left, "right": right,
            "up_left": up_left, "up_right": up_right,
            "down_left": down_left, "down_right": down_right,
        }
    return None


centroid = coords_arr.mean(axis=0)
dist_to_centroid = np.hypot(coords_arr[:, 0] - centroid[0], coords_arr[:, 1] - centroid[1])
order = np.argsort(dist_to_centroid)
n_candidates = max(N_NEIGHBORHOODS * 9, int(round(len(cells_fov_ids) * CENTRAL_CANDIDATE_FRACTION)))
central_candidates = [cells_fov_ids[i] for i in order[:n_candidates]]

rng = np.random.default_rng(SEED)
shuffled = list(central_candidates)
rng.shuffle(shuffled)

used_fovs = set()
neighborhoods = []
for candidate in shuffled:
    if candidate in used_fovs:
        continue
    block = find_3x3_block([candidate], full_positions, STEP_SIZE_UM, TOLERANCE_FRACTION)
    if block is None:
        continue
    block_fovs = set(block.values())
    if block_fovs & used_fovs:
        continue
    neighborhoods.append(block)
    used_fovs |= block_fovs
    if len(neighborhoods) == N_NEIGHBORHOODS:
        break

if len(neighborhoods) < N_NEIGHBORHOODS:
    raise RuntimeError(
        f"Only found {len(neighborhoods)}/{N_NEIGHBORHOODS} independent 3x3 neighbourhoods "
        f"near the tissue centroid -- try raising CENTRAL_CANDIDATE_FRACTION.")

for i, block in enumerate(neighborhoods):
    print(f"Neighbourhood {i}: center={block['center']}  {block}")

## 6 — Overview: the 3 neighbourhoods on the full FOV grid

In [ ]:
figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

NEIGHBORHOOD_COLORS = ["tab:red", "tab:blue", "tab:green", "tab:orange", "tab:purple"]

fig, ax = plt.subplots(figsize=(8, 8))
all_xy = np.array([full_positions[f] for f in cells_fov_ids])
ax.scatter(all_xy[:, 0], all_xy[:, 1], s=4, color="0.8", label="all cells-round FOVs", zorder=1)
ax.scatter(*centroid, marker="x", s=120, color="k", label="tissue centroid", zorder=4)

for i, block in enumerate(neighborhoods):
    color = NEIGHBORHOOD_COLORS[i % len(NEIGHBORHOOD_COLORS)]
    xy = np.array([full_positions[f] for f in block.values()])
    ax.scatter(xy[:, 0], xy[:, 1], s=40, color=color, zorder=3,
               label=f"neighbourhood {i} (center={block['center']})")
    cx, cy = full_positions[block["center"]]
    ax.scatter([cx], [cy], s=90, facecolors="none", edgecolors=color, linewidths=2, zorder=3)

ax.invert_yaxis()
ax.axis("equal")
ax.set_xlabel("x (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("y (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"{SAMPLE_NAME}: 3 independent 3x3 neighbourhoods", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE, loc="best")
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.overview.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")

## 7 — Registration helpers

- `register_cardinal`: registers a block's 4 cardinal neighbours directly to
  its centre (`register_neighbor_pair`, same primitive the correction
  pipeline uses).
- `corner_check`: at one corner (e.g. up-left), crops the two overlap bands
  that meet there (`crop_overlap` with the "up" and "left" directions), the
  intersection of which IS the corner region shared by all three FOVs. Each
  neighbour's own overlap-band crop is sub-pixel shifted onto the centre's
  frame using the pixel shift implied by its OWN direct registration above,
  then the two aligned corners are compared directly (`phase_drift`) -- if
  the grid is rigid these should coincide; the residual is exactly how much
  they don't.
- `register_chained`: registers *target_img* against *anchor_img* (whose own
  measured position may differ from its nominal grid position), returning
  the target's position CHAIN-composed through the anchor's own measured
  position (not just the raw pairwise registration).
- `diagonal_loop_closure`: for one corner's diagonal FOV, chains through
  BOTH of that corner's already-registered cardinal neighbours independently
  and compares the two resulting positions -- BigStitcher's loop-closure
  check.

In [ ]:
CORNER_DIRS = {
    "up_left":    ("up", "left"),
    "up_right":   ("up", "right"),
    "down_left":  ("down", "left"),
    "down_right": ("down", "right"),
}
DIAGONAL_CHAIN = {
    "up_left":    (("up", "left"), ("left", "up")),
    "up_right":   (("up", "right"), ("right", "up")),
    "down_left":  (("down", "left"), ("left", "down")),
    "down_right": (("down", "right"), ("right", "down")),
}


def register_cardinal(block, frames, channel):
    center_img = frames[block["center"]][channel]
    out = {}
    for direction in ("up", "down", "left", "right"):
        nb_fov = block[direction]
        nb_img = frames[nb_fov][channel]
        measured_xy, error = register_neighbor_pair(
            center_img, nb_img, full_positions[block["center"]], full_positions[nb_fov],
            direction, OVERLAP_FRACTION, PIXEL_SIZE_UM, UPSAMPLE_FACTOR,
        )
        out[direction] = {
            "neighbor_fov": nb_fov, "nominal_xy": full_positions[nb_fov],
            "measured_xy": measured_xy, "error": error,
        }
    return out


def _pixel_shift_from_measured(measured_xy, nominal_xy):
    '''Back-solve the (dy_px, dx_px) phase-correlation shift register_neighbor_pair
    measured, from its own returned measured_xy -- measured_xy is always exactly
    nominal_xy + (dx_px, dy_px) * PIXEL_SIZE_UM (register_neighbor_pair's own algebra),
    so this recovers it without re-running phase_drift.'''
    dx_px = (measured_xy[0] - nominal_xy[0]) / PIXEL_SIZE_UM
    dy_px = (measured_xy[1] - nominal_xy[1]) / PIXEL_SIZE_UM
    return dy_px, dx_px


def corner_check(block, frames, channel, cardinal_results, corner_name):
    dir1, dir2 = CORNER_DIRS[corner_name]   # dir1 in {up,down}, dir2 in {left,right}
    center_img = frames[block["center"]][channel]
    img1, img2 = frames[block[dir1]][channel], frames[block[dir2]][channel]

    a1, n1 = crop_overlap(center_img, img1, dir1, OVERLAP_FRACTION)   # a1,n1 shape (N, w)
    a2, n2 = crop_overlap(center_img, img2, dir2, OVERLAP_FRACTION)   # a2,n2 shape (h, N)
    N = a1.shape[0]

    dy1, dx1 = _pixel_shift_from_measured(cardinal_results[dir1]["measured_xy"], cardinal_results[dir1]["nominal_xy"])
    dy2, dx2 = _pixel_shift_from_measured(cardinal_results[dir2]["measured_xy"], cardinal_results[dir2]["nominal_xy"])
    aligned1 = ndi_shift(n1.astype(float), shift=(dy1, dx1), order=1, mode="nearest")
    aligned2 = ndi_shift(n2.astype(float), shift=(dy2, dx2), order=1, mode="nearest")

    # a1's own row-range is crop_overlap's "up"/"down" convention: "up" takes the
    # anchor's LAST N rows, "down" takes its FIRST N rows (see crop_overlap's own
    # docstring) -- row_slice must select that SAME range back out of a2 (which
    # spans all h rows), so it's the OPPOSITE mapping from a naive dir1=="up"->first-N read.
    col_slice = slice(0, N) if dir2 == "left"  else slice(-N, None)
    row_slice = slice(-N, None) if dir1 == "up" else slice(0, N)

    center_corner   = a1[:, col_slice]
    corner_from_dir1 = aligned1[:, col_slice]
    corner_from_dir2 = aligned2[row_slice, :]

    # Sanity check -- both are literal slices of the SAME center_img, must match exactly.
    consistency_check = float(np.max(np.abs(center_corner.astype(float) - a2[row_slice, :].astype(float))))

    shift_px, err = phase_drift(remove_hot_pixels(corner_from_dir1), remove_hot_pixels(corner_from_dir2), UPSAMPLE_FACTOR)
    residual_um = float(np.hypot(*shift_px) * PIXEL_SIZE_UM)

    return {
        "residual_um": residual_um, "registration_error": err,
        "center_slice_consistency_maxdiff": consistency_check,
        "center_corner": center_corner, "corner_from_dir1": corner_from_dir1, "corner_from_dir2": corner_from_dir2,
        "dir1": dir1, "dir2": dir2,
    }


def register_chained(anchor_measured_xy, anchor_nominal_xy, anchor_img, target_img, target_nominal_xy, direction):
    raw_measured_xy, error = register_neighbor_pair(
        anchor_img, target_img, anchor_nominal_xy, target_nominal_xy,
        direction, OVERLAP_FRACTION, PIXEL_SIZE_UM, UPSAMPLE_FACTOR,
    )
    relative_offset = (raw_measured_xy[0] - anchor_nominal_xy[0], raw_measured_xy[1] - anchor_nominal_xy[1])
    chained_xy = (anchor_measured_xy[0] + relative_offset[0], anchor_measured_xy[1] + relative_offset[1])
    return chained_xy, error


def diagonal_loop_closure(block, frames, channel, cardinal_results, corner_name):
    (anchor1_key, dir1), (anchor2_key, dir2) = DIAGONAL_CHAIN[corner_name]
    diag_fov = block[corner_name]
    diag_img = frames[diag_fov][channel]

    anchor1_fov, anchor2_fov = block[anchor1_key], block[anchor2_key]
    via1_xy, err1 = register_chained(
        cardinal_results[anchor1_key]["measured_xy"], full_positions[anchor1_fov],
        frames[anchor1_fov][channel], diag_img, full_positions[diag_fov], dir1,
    )
    via2_xy, err2 = register_chained(
        cardinal_results[anchor2_key]["measured_xy"], full_positions[anchor2_fov],
        frames[anchor2_fov][channel], diag_img, full_positions[diag_fov], dir2,
    )
    residual_um = float(np.hypot(via1_xy[0] - via2_xy[0], via1_xy[1] - via2_xy[1]))
    return {
        "diag_fov": diag_fov, "anchor1_key": anchor1_key, "anchor2_key": anchor2_key,
        "via1_xy": via1_xy, "via2_xy": via2_xy, "error1": err1, "error2": err2,
        "residual_um": residual_um,
    }

## 8 — Run the pipeline (calculation, cached)

Loads all 9 FOVs' bead + DAPI frames per neighbourhood, then runs 4-connected
registration, corner concordance, and diagonal loop closure on both channels.
Cached to `analysis/cache/test_stitching/*.csv` (`NOTEBOOK_GUIDELINES.md`
#2/#3) -- re-running the notebook skips straight to display unless
`FORCE_RECOMPUTE=True`.

In [ ]:
CACHE_DIR = config.analysis_dir / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
cardinal_csv = CACHE_DIR / "cardinal_results.csv"
corner_csv   = CACHE_DIR / "corner_results.csv"
diagonal_csv = CACHE_DIR / "diagonal_results.csv"

if not FORCE_RECOMPUTE and cardinal_csv.exists() and corner_csv.exists() and diagonal_csv.exists():
    cardinal_df = pd.read_csv(cardinal_csv)
    corner_df   = pd.read_csv(corner_csv)
    diagonal_df = pd.read_csv(diagonal_csv)
    corner_crops = {}   # not cached (image data) -- regenerate on demand for display below if needed
    print(f"Loaded cached results from {CACHE_DIR}")
else:
    cardinal_rows, corner_rows, diagonal_rows = [], [], []
    corner_crops = {}   # {(neighborhood_id, channel, corner_name): corner_check() result}

    reporter = ProgressReporter(total=len(neighborhoods) * 9, label="Loading FOV frames")
    all_frames = []   # all_frames[i] = {fov_id: {"beads": img, "dapi": img}}
    for block in neighborhoods:
        frames = {}
        for fov_id in sorted(set(block.values())):
            frames[fov_id] = load_channel_frames(fov_id)
            reporter.update(1)
        all_frames.append(frames)
    reporter.done()

    for i, (block, frames) in enumerate(zip(neighborhoods, all_frames)):
        for channel in CHANNELS:
            cardinal_results = register_cardinal(block, frames, channel)
            for direction, r in cardinal_results.items():
                cardinal_rows.append({
                    "neighborhood": i, "channel": channel, "direction": direction,
                    "center_fov": block["center"], "neighbor_fov": r["neighbor_fov"],
                    "nominal_x": r["nominal_xy"][0], "nominal_y": r["nominal_xy"][1],
                    "measured_x": r["measured_xy"][0], "measured_y": r["measured_xy"][1],
                    "shift_um": float(np.hypot(r["measured_xy"][0] - r["nominal_xy"][0],
                                                r["measured_xy"][1] - r["nominal_xy"][1])),
                    "error": r["error"],
                })

            for corner_name in CORNER_DIRS:
                cc = corner_check(block, frames, channel, cardinal_results, corner_name)
                corner_crops[(i, channel, corner_name)] = cc
                corner_rows.append({
                    "neighborhood": i, "channel": channel, "corner": corner_name,
                    "center_fov": block["center"],
                    "residual_um": cc["residual_um"], "registration_error": cc["registration_error"],
                    "center_slice_consistency_maxdiff": cc["center_slice_consistency_maxdiff"],
                })

            for corner_name in DIAGONAL_CHAIN:
                dc = diagonal_loop_closure(block, frames, channel, cardinal_results, corner_name)
                diagonal_rows.append({
                    "neighborhood": i, "channel": channel, "corner": corner_name,
                    "diag_fov": dc["diag_fov"], "anchor1_key": dc["anchor1_key"], "anchor2_key": dc["anchor2_key"],
                    "via1_x": dc["via1_xy"][0], "via1_y": dc["via1_xy"][1],
                    "via2_x": dc["via2_xy"][0], "via2_y": dc["via2_xy"][1],
                    "error1": dc["error1"], "error2": dc["error2"],
                    "residual_um": dc["residual_um"],
                })

    cardinal_df = pd.DataFrame(cardinal_rows)
    corner_df   = pd.DataFrame(corner_rows)
    diagonal_df = pd.DataFrame(diagonal_rows)
    cardinal_df.to_csv(cardinal_csv, index=False)
    corner_df.to_csv(corner_csv, index=False)
    diagonal_df.to_csv(diagonal_csv, index=False)
    print(f"Saved: {cardinal_csv}, {corner_csv}, {diagonal_csv}")

print(f"\ncardinal_df: {len(cardinal_df)} rows, corner_df: {len(corner_df)} rows, "
      f"diagonal_df: {len(diagonal_df)} rows")

## 9 — 4-connected registration results, beads vs. DAPI

Per-neighbourhood, per-direction shift magnitude (nominal vs. measured
position), plus a direct beads-vs-DAPI comparison: do the two independent
channels agree on the same real neighbour-to-neighbour shift?

In [ ]:
print(cardinal_df.pivot_table(index=["neighborhood", "direction"], columns="channel",
                              values="shift_um").round(3))

beads_wide = cardinal_df[cardinal_df.channel == "beads"].set_index(["neighborhood", "direction"])
dapi_wide  = cardinal_df[cardinal_df.channel == "dapi"].set_index(["neighborhood", "direction"])
channel_agreement = pd.DataFrame({
    "beads_measured_x": beads_wide["measured_x"], "beads_measured_y": beads_wide["measured_y"],
    "dapi_measured_x": dapi_wide["measured_x"], "dapi_measured_y": dapi_wide["measured_y"],
})
channel_agreement["beads_vs_dapi_um"] = np.hypot(
    channel_agreement["beads_measured_x"] - channel_agreement["dapi_measured_x"],
    channel_agreement["beads_measured_y"] - channel_agreement["dapi_measured_y"],
)
print("\nBeads vs. DAPI agreement on the same neighbour-to-neighbour registration:")
print(channel_agreement["beads_vs_dapi_um"].round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
labels = [f"nb{n}-{d}" for n, d in channel_agreement.index]
ax.bar(labels, channel_agreement["beads_vs_dapi_um"], color="tab:purple")
ax.set_ylabel("|beads - DAPI| measured position (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"{SAMPLE_NAME}: beads vs. DAPI agreement per 4-connected registration",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(axis="x", labelsize=PLOT_TICK_FONTSIZE - 2, rotation=45)
ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.beads_vs_dapi.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")

## 10 — Corner concordance

`residual_um` is how much the corner region disagrees depending on which of
the two adjacent 4-connected neighbours' alignment you trust -- 0 would mean
perfect grid rigidity. `center_slice_consistency_maxdiff` is a pure sanity
check (should be exactly 0.0 -- both `center_corner` extractions are literal
slices of the same image).

In [ ]:
print(corner_df.pivot_table(index=["neighborhood", "corner"], columns="channel",
                            values="residual_um").round(3))
print(f"\nMax center-slice consistency check (should be 0.0): "
      f"{corner_df['center_slice_consistency_maxdiff'].max()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pivot = corner_df.pivot_table(index=["neighborhood", "corner"], columns="channel", values="residual_um")
x = np.arange(len(pivot))
width = 0.35
for offset, channel in zip((-width/2, width/2), CHANNELS):
    ax.bar(x + offset, pivot[channel], width=width, label=channel)
ax.set_xticks(x)
ax.set_xticklabels([f"nb{n}\n{c}" for n, c in pivot.index], fontsize=PLOT_TICK_FONTSIZE - 3)
ax.set_ylabel("corner residual (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"{SAMPLE_NAME}: corner concordance residual (0 = perfectly consistent)",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.corner_residuals.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")

### Corner overlay pictures

Per neighbourhood, per channel: the corner region as implied by each of the
two adjacent neighbours' own alignment (red / green), plus the centre FOV's
own true corner pixels (blue) for reference -- well-aligned corners blend
toward white/gray; a real disagreement shows as separated red/green
ghosting. Regenerated on demand from `neighborhoods`/`all_frames` (not
cached -- image arrays, not scalars) if a cache was loaded above.

In [ ]:
def _normalize01(img):
    lo, hi = np.percentile(img, [1, 99.5])
    return np.clip((img.astype(np.float32) - lo) / max(hi - lo, 1e-6), 0, 1)


def corner_rgb_overlay(cc):
    return np.stack([_normalize01(cc["corner_from_dir1"]), _normalize01(cc["corner_from_dir2"]),
                      _normalize01(cc["center_corner"])], axis=-1)


if not corner_crops:
    # Cache was loaded from disk above with no image data -- recompute just the crops for display.
    reporter = ProgressReporter(total=len(neighborhoods) * 9, label="Reloading FOV frames for display")
    all_frames = []
    for block in neighborhoods:
        frames = {}
        for fov_id in sorted(set(block.values())):
            frames[fov_id] = load_channel_frames(fov_id)
            reporter.update(1)
        all_frames.append(frames)
    reporter.done()
    for i, (block, frames) in enumerate(zip(neighborhoods, all_frames)):
        for channel in CHANNELS:
            cardinal_results = register_cardinal(block, frames, channel)
            for corner_name in CORNER_DIRS:
                corner_crops[(i, channel, corner_name)] = corner_check(block, frames, channel, cardinal_results, corner_name)

for i in range(len(neighborhoods)):
    for channel in CHANNELS:
        fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
        for ax, corner_name in zip(axes, CORNER_DIRS):
            cc = corner_crops[(i, channel, corner_name)]
            ax.imshow(corner_rgb_overlay(cc))
            ax.set_title(f"{corner_name} (red={cc['dir1']}, green={cc['dir2']})\n"
                         f"residual={cc['residual_um']:.3f} um", fontsize=PLOT_TICK_FONTSIZE - 1)
            ax.axis("off")
        # dir1/dir2 vary PER CORNER (up_left uses up/left, up_right uses up/right, ...) -- each
        # panel's own title above states which cardinal directions red/green actually are for
        # that corner; this legend only needs to explain the fixed role each color plays.
        handles = [mpatches.Patch(color=c, label=n) for n, c in
                   (("vertical neighbour (up or down)", "red"),
                    ("horizontal neighbour (left or right)", "green"),
                    ("center (reference)", "blue"))]
        fig.legend(handles=handles, loc="lower center", ncol=3, fontsize=PLOT_LEGEND_FONTSIZE)
        fig.suptitle(f"{SAMPLE_NAME}: neighbourhood {i} (center={neighborhoods[i]['center']}) -- "
                     f"{channel} corner overlays", fontsize=PLOT_TITLE_FONTSIZE)
        fig.tight_layout(rect=[0, 0.08, 1, 0.93])
        fig_path = figures_dir / f"{NOTEBOOK_NAME}.corners_nb{i}_{channel}.png"
        fig.savefig(fig_path, dpi=150)
        plt.show()
        print(f"Saved: {fig_path}")

## 11 — Diagonal loop closure

For each corner's diagonal FOV, `residual_um` is the disagreement between
its position as registered via the "up"-side chain vs. the "left"-side
chain (or the equivalent pair for the other 3 corners) -- BigStitcher's
classic loop-closure consistency check: a rigid, consistent grid gives the
same answer either way.

In [ ]:
print(diagonal_df.pivot_table(index=["neighborhood", "corner"], columns="channel",
                              values="residual_um").round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
pivot = diagonal_df.pivot_table(index=["neighborhood", "corner"], columns="channel", values="residual_um")
x = np.arange(len(pivot))
width = 0.35
for offset, channel in zip((-width/2, width/2), CHANNELS):
    ax.bar(x + offset, pivot[channel], width=width, label=channel)
ax.set_xticks(x)
ax.set_xticklabels([f"nb{n}\n{c}" for n, c in pivot.index], fontsize=PLOT_TICK_FONTSIZE - 3)
ax.set_ylabel("loop-closure residual (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"{SAMPLE_NAME}: diagonal loop-closure residual (0 = perfectly consistent)",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(axis="y", labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()
fig_path = figures_dir / f"{NOTEBOOK_NAME}.diagonal_residuals.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")

### Diagonal overlay pictures

Places the two anchor FOVs at their own measured positions (blue / cyan),
and the diagonal FOV TWICE, tinted red at its "via anchor 1"-chained
position and green at its "via anchor 2"-chained position -- a consistent
grid places both diagonal copies on top of each other (yellow-ish where they
overlap); a real loop-closure disagreement shows as a visible red/green
double-image.

In [ ]:
def place_and_blend(frames_dict, positions_um, colors):
    h, w = next(iter(frames_dict.values())).shape
    xs = [positions_um[n][0] for n in frames_dict]
    ys = [positions_um[n][1] for n in frames_dict]
    x_min = min(xs) - w * PIXEL_SIZE_UM / 2
    y_min = min(ys) - h * PIXEL_SIZE_UM / 2

    # Derive canvas size from the SAME rounded per-tile row0/col0 used to place
    # each tile (rather than independently rounding (x_max-x_min)/PIXEL_SIZE_UM)
    # -- otherwise the two roundings can disagree by a pixel and the canvas ends
    # up one pixel too small for a tile placed at the far edge.
    placements = {}
    for name in frames_dict:
        x_um, y_um = positions_um[name]
        col0 = int(round((x_um - w * PIXEL_SIZE_UM / 2 - x_min) / PIXEL_SIZE_UM))
        row0 = int(round((y_um - h * PIXEL_SIZE_UM / 2 - y_min) / PIXEL_SIZE_UM))
        placements[name] = (row0, col0)
    canvas_h = max(row0 + h for row0, col0 in placements.values())
    canvas_w = max(col0 + w for row0, col0 in placements.values())

    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.float32)
    for name, img in frames_dict.items():
        norm = _normalize01(img)
        row0, col0 = placements[name]
        canvas[row0:row0 + h, col0:col0 + w] += norm[..., None] * np.array(colors[name], dtype=np.float32)
    return np.clip(canvas, 0, 1), x_min, y_min


def crop_at_um(canvas, x_min, y_min, x_um, y_um, half_um):
    row = int(round((y_um - y_min) / PIXEL_SIZE_UM))
    col = int(round((x_um - x_min) / PIXEL_SIZE_UM))
    half_px = int(round(half_um / PIXEL_SIZE_UM))
    h, w = canvas.shape[:2]
    r0, r1 = max(row - half_px, 0), min(row + half_px, h)
    c0, c1 = max(col - half_px, 0), min(col + half_px, w)
    return canvas[r0:r1, c0:c1]


# Wide enough to also show a margin of BOTH anchor tiles for context, not just
# the diagonal FOV's own interior -- the anchors sit roughly STEP_SIZE_UM away
# from the diagonal's own centre, so a narrow zoom (e.g. FRAME_WIDTH_UM/4) never
# reaches far enough to include any of their content at all.
DIAGONAL_ZOOM_HALF_UM = FRAME_WIDTH_UM * 0.6
DIAGONAL_COLORS = {"anchor1": (0.2, 0.2, 1.0), "anchor2": (0.2, 0.8, 0.8),
                   "diag_via1": (1.0, 0.0, 0.0), "diag_via2": (0.0, 1.0, 0.0)}

for i, block in enumerate(neighborhoods):
    for channel in CHANNELS:
        frames = all_frames[i]
        cardinal_results = register_cardinal(block, frames, channel)
        fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
        for ax, corner_name in zip(axes, DIAGONAL_CHAIN):
            dc_row = diagonal_df[(diagonal_df.neighborhood == i) & (diagonal_df.channel == channel)
                                  & (diagonal_df.corner == corner_name)].iloc[0]
            anchor1_key, anchor2_key = dc_row["anchor1_key"], dc_row["anchor2_key"]
            anchor1_fov, anchor2_fov = block[anchor1_key], block[anchor2_key]
            diag_fov = block[corner_name]
            place_frames = {
                "anchor1": frames[anchor1_fov][channel], "anchor2": frames[anchor2_fov][channel],
                "diag_via1": frames[diag_fov][channel], "diag_via2": frames[diag_fov][channel],
            }
            place_positions = {
                "anchor1": cardinal_results[anchor1_key]["measured_xy"],
                "anchor2": cardinal_results[anchor2_key]["measured_xy"],
                "diag_via1": (dc_row["via1_x"], dc_row["via1_y"]),
                "diag_via2": (dc_row["via2_x"], dc_row["via2_y"]),
            }
            canvas, x_min, y_min = place_and_blend(place_frames, place_positions, DIAGONAL_COLORS)
            zoom_x = (dc_row["via1_x"] + dc_row["via2_x"]) / 2
            zoom_y = (dc_row["via1_y"] + dc_row["via2_y"]) / 2
            zoom = crop_at_um(canvas, x_min, y_min, zoom_x, zoom_y, DIAGONAL_ZOOM_HALF_UM)
            ax.imshow(zoom)
            ax.set_title(f"{corner_name}\nresidual={dc_row['residual_um']:.3f} um", fontsize=PLOT_TICK_FONTSIZE)
            ax.axis("off")
        handles = [mpatches.Patch(color=c, label=n) for n, c in DIAGONAL_COLORS.items()]
        fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=PLOT_LEGEND_FONTSIZE)
        fig.suptitle(f"{SAMPLE_NAME}: neighbourhood {i} (center={block['center']}) -- "
                     f"{channel} diagonal loop-closure overlays", fontsize=PLOT_TITLE_FONTSIZE)
        fig.tight_layout(rect=[0, 0.08, 1, 0.93])
        fig_path = figures_dir / f"{NOTEBOOK_NAME}.diagonals_nb{i}_{channel}.png"
        fig.savefig(fig_path, dpi=150)
        plt.show()
        print(f"Saved: {fig_path}")